# 06 — Socioeconomic

Fetches Census ACS-5 socioeconomic indicators directly per census tract. Since our unit of analysis is already census tracts, no spatial matching is needed — direct FIPS code join.

**Data source:** US Census ACS-5 2022 API — portable to all US cities.

**Output columns:** `tract_id`, `median_income`, `population_density`, `poverty_rate`

**Output file:** `csv/06_socioeconomic.csv`

In [ ]:
ZONES_CONFIG = "zones.json"

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import os

os.makedirs("csv", exist_ok=True)

with open(ZONES_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

BOROUGH_FILTER = config["borough_filter"]
BOROUGH_CODES = config["borough_codes"]

# NYC FIPS county codes
BORO_TO_COUNTY = {
    "MN": "061",  # Manhattan (New York County)
    "BX": "005",  # Bronx
    "BK": "047",  # Brooklyn (Kings County)
    "QN": "081",  # Queens
    "SI": "085",  # Staten Island (Richmond County)
}

counties = [BORO_TO_COUNTY[b] for b in BOROUGH_FILTER]

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")
print(f"Counties to query: {counties}")

In [ ]:
# ── Query Census ACS-5 API ────────────────────────────
#
# Variables:
#   B19013_001E  = Median household income
#   B01003_001E  = Total population
#   B17001_002E  = Population below poverty level
#   B17001_001E  = Population for whom poverty status determined

ACS_VARS = "B19013_001E,B01003_001E,B17001_002E,B17001_001E,NAME"
STATE_FIPS = "36"  # New York

all_tracts = []

for county in counties:
    url = "https://api.census.gov/data/2022/acs/acs5"
    params = {
        "get": ACS_VARS,
        "for": "tract:*",
        "in": f"state:{STATE_FIPS} county:{county}",
    }
    try:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        chunk = pd.DataFrame(data[1:], columns=data[0])
        all_tracts.append(chunk)
        print(f"  County {county}: {len(chunk)} tracts")
    except Exception as e:
        print(f"  County {county}: FAILED — {e}")

df_census = pd.concat(all_tracts, ignore_index=True)
print(f"Total Census tracts: {len(df_census)}")

In [ ]:
# ── Process Census data ───────────────────────────────

# Build tract_id matching PLUTO's bct2020 format: borocode + tract
# PLUTO bct2020 = borocode(1 digit) + tract(6 digits, zero-padded)
# Census gives: state(2) + county(3) + tract(6)
# For Manhattan: borocode=1, county=061

COUNTY_TO_BORO = {v: str(BOROUGH_CODES[k]) for k, v in BORO_TO_COUNTY.items() if k in BOROUGH_FILTER}

df_census["tract_id"] = df_census.apply(
    lambda row: COUNTY_TO_BORO.get(row["county"], "0") + row["tract"].zfill(6),
    axis=1
)

# Convert numeric columns
df_census["median_income"] = pd.to_numeric(df_census["B19013_001E"], errors="coerce")
df_census["total_pop"] = pd.to_numeric(df_census["B01003_001E"], errors="coerce")
df_census["poverty_pop"] = pd.to_numeric(df_census["B17001_002E"], errors="coerce")
df_census["poverty_universe"] = pd.to_numeric(df_census["B17001_001E"], errors="coerce")

# Census uses negative values as sentinel for missing
df_census.loc[df_census["median_income"] < 0, "median_income"] = np.nan
df_census.loc[df_census["total_pop"] < 0, "total_pop"] = np.nan

# Poverty rate
df_census["poverty_rate"] = np.where(
    df_census["poverty_universe"] > 0,
    (df_census["poverty_pop"] / df_census["poverty_universe"] * 100).round(1),
    np.nan
)

# Population density (per km2, using approximate tract area)
# Manhattan is ~59 km2 with ~309 tracts → ~0.19 km2/tract average
AVG_TRACT_AREA_KM2 = 0.19
df_census["population_density"] = (df_census["total_pop"] / AVG_TRACT_AREA_KM2).round(0)

print(f"Tracts with income data: {df_census['median_income'].notna().sum()}/{len(df_census)}")
print(f"Income range: ${df_census['median_income'].min():,.0f} – ${df_census['median_income'].max():,.0f}")

In [ ]:
# ── Join to our tracts ────────────────────────────────

df_result = df_tracts[["tract_id"]].merge(
    df_census[["tract_id", "median_income", "population_density", "poverty_rate"]],
    on="tract_id",
    how="left"
)

matched = df_result["median_income"].notna().sum()
print(f"Matched: {matched}/{len(df_result)} tracts")
if matched < len(df_result):
    missing = df_result[df_result["median_income"].isna()]["tract_id"].tolist()
    print(f"  Missing tracts: {missing[:10]}{'...' if len(missing) > 10 else ''}")

print(f"\nSummary:")
print(df_result[["median_income", "population_density", "poverty_rate"]].describe().round(1).to_string())

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/06_socioeconomic.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)